In [5]:
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import os
import glob
import pandas as pd
import datetime
import numpy as np
import polars as pl
from pathlib import Path

# 주제 1. 국내 채용시장 및 채용 플랫폼 이용패턴 분석

# Company.csv
- 기업 마스터 테이블
- 기업id, 기업 생성일, 마지막 기업 정보 수정일, 설립일, 구성원 수, 기업 팔로워 수, 기업 추천 수


| 컬럼명            | 데이터 타입 | 설명                              |
|-------------------|-------------|-----------------------------------|
| id                | INTEGER     | 기업 id (PK, company_id)         |
| cdate             | DATETIME    | 기업 생성일(UTC), 플랫폼에 기업 정보 등록한 시점 |
| mdate             | DATETIME    | 마지막 기업정보 수정일           |
| found_date        | DATE        | 설립일(KST)                       |
| employee_count    | STRING      | 구성원 수, 데이터 수집 시점에 집계된 숫자 |
| view_count        | INTEGER     | 기업 정보 조회수, 데이터 수집 시점에 집계된 숫자 |
| follow_count      | INTEGER     | 기업 팔로우 수, 데이터 수집 시점에 집계된 숫자 |
| reference_count   | INTEGER     | 기업 추천 수, 데이터 수집 시점에 집계된 숫자 |


In [13]:
DATA_DIR = Path(r"C:\Users\user\Desktop\playground\codeit_DA_13th\projects\intermediate1\midproject1\DBsources\topic1")
# 데이터 모아둔 폴더 경로

company = pl.read_parquet(DATA_DIR / 'company.parquet')

company = company.with_columns(
    pl.col(pl.Utf8).replace(["", "nan", "NaN"], None)
)

In [15]:
company.select(pl.all().null_count())

cdate,mdate,found_date,employee_count,view_count,follow_count,reference_count,company_uuid
u32,u32,u32,u32,u32,u32,u32,u32
0,0,24175,0,0,0,0,0


설립일 결측치가 많다

In [16]:
company.select(pl.all().n_unique())

cdate,mdate,found_date,employee_count,view_count,follow_count,reference_count,company_uuid
u32,u32,u32,u32,u32,u32,u32,u32
41329,35488,5200,8,6830,458,63,41659


기업의 uuid는 모두 유니크하다

In [19]:
# 개수(count)가 많은 순서대로 정렬해서 보기
company['employee_count'].value_counts(sort=True)

employee_count,count
str,u32
"""0명""",34978
"""1-10명""",3202
"""11-50명""",2589
"""51-200명""",673
"""201-500명""",125
"""501-1000명""",42
"""1001-5000명""",40
"""5000명 초과""",10


사원수는 0명- 아마 미수집 데이터 일듯한데 가장 많다

1. 전체 데이터
2. 0배제
3. 0만

순서로 분석 진행

In [41]:
# 사원수 0명 데이터 배제
filtered_company = company.filter(pl.col("employee_count") != "0명")
# 사원수 0명 데이터만
employee_zero_company = company.filter(pl.col("employee_count") == "0명")

print(employee_zero_company.head())

shape: (5, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ cdate     ┆ mdate     ┆ found_dat ┆ employee_ ┆ … ┆ follow_co ┆ reference ┆ company_u ┆ company_ │
│ ---       ┆ ---       ┆ e         ┆ count     ┆   ┆ unt       ┆ _count    ┆ uid       ┆ segment  │
│ datetime[ ┆ datetime[ ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ μs]       ┆ μs]       ┆ str       ┆ str       ┆   ┆ i32       ┆ i32       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2022-06-0 ┆ 2022-06-0 ┆ null      ┆ 0명       ┆ … ┆ 0         ┆ 0         ┆ c46f2fa5- ┆ 일반     │
│ 9         ┆ 9         ┆           ┆           ┆   ┆           ┆           ┆ f940-40fb ┆ 기업     │
│ 04:08:40  ┆ 04:08:40  ┆           ┆           ┆   ┆           ┆           ┆ -bd4a-aa0 ┆          │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ ff7…

### 조회수, 팔로우 수, 추천 수

In [59]:
result = (
    company.select(['view_count', 'follow_count', 'reference_count'])
    .describe()
    .filter(pl.col("statistic").is_in(['mean', 'max', 'min', '50%']))
    .with_columns(pl.all().exclude("statistic").cast(pl.Float64).round(1))
)
print(result)

shape: (4, 4)
┌───────────┬────────────┬──────────────┬─────────────────┐
│ statistic ┆ view_count ┆ follow_count ┆ reference_count │
│ ---       ┆ ---        ┆ ---          ┆ ---             │
│ str       ┆ f64        ┆ f64          ┆ f64             │
╞═══════════╪════════════╪══════════════╪═════════════════╡
│ mean      ┆ 1661.8     ┆ 10.2         ┆ 0.7             │
│ min       ┆ 0.0        ┆ 0.0          ┆ 0.0             │
│ 50%       ┆ 536.0      ┆ 0.0          ┆ 0.0             │
│ max       ┆ 575596.0   ┆ 2740.0       ┆ 249.0           │
└───────────┴────────────┴──────────────┴─────────────────┘


In [60]:
# 사원수 0명 배제시 
result = (
    filtered_company.select(['view_count', 'follow_count', 'reference_count'])
    .describe()
    .filter(pl.col("statistic").is_in(['mean', 'max', 'min', '50%']))
    .with_columns(pl.all().exclude("statistic").cast(pl.Float64).round(1))
)
print(result)

shape: (4, 4)
┌───────────┬────────────┬──────────────┬─────────────────┐
│ statistic ┆ view_count ┆ follow_count ┆ reference_count │
│ ---       ┆ ---        ┆ ---          ┆ ---             │
│ str       ┆ f64        ┆ f64          ┆ f64             │
╞═══════════╪════════════╪══════════════╪═════════════════╡
│ mean      ┆ 4865.0     ┆ 44.5         ┆ 3.1             │
│ min       ┆ 4.0        ┆ 0.0          ┆ 0.0             │
│ 50%       ┆ 2585.0     ┆ 9.0          ┆ 1.0             │
│ max       ┆ 575596.0   ┆ 2740.0       ┆ 249.0           │
└───────────┴────────────┴──────────────┴─────────────────┘


In [ ]:
# 사원수 0명 only
result = (
    employee_zero_company.select(['view_count', 'follow_count', 'reference_count'])
    .describe()
    .filter(pl.col("statistic").is_in(['mean', 'max', 'min', '50%']))
    .with_columns(pl.all().exclude("statistic").cast(pl.Float64).round(1))
)
print(result)

shape: (4, 4)
┌───────────┬────────────┬──────────────┬─────────────────┐
│ statistic ┆ view_count ┆ follow_count ┆ reference_count │
│ ---       ┆ ---        ┆ ---          ┆ ---             │
│ str       ┆ f64        ┆ f64          ┆ f64             │
╞═══════════╪════════════╪══════════════╪═════════════════╡
│ mean      ┆ 1049.9     ┆ 3.6          ┆ 0.3             │
│ min       ┆ 0.0        ┆ 0.0          ┆ 0.0             │
│ 50%       ┆ 379.0      ┆ 0.0          ┆ 0.0             │
│ max       ┆ 180132.0   ┆ 2056.0       ┆ 90.0            │
└───────────┴────────────┴──────────────┴─────────────────┘


### 인기 그룹 확인

In [57]:
# 상위 10% (0.9)와 상위 5% (0.95) 지점 확인
thresholds = company.select([
    pl.col("view_count").quantile(0.95).alias("view_95p"),
    pl.col("follow_count").quantile(0.95).alias("follow_95p"),
    pl.col("reference_count").quantile(0.95).alias("reference_95p")
])
print(thresholds)

shape: (1, 3)
┌──────────┬────────────┬───────────────┐
│ view_95p ┆ follow_95p ┆ reference_95p │
│ ---      ┆ ---        ┆ ---           │
│ f64      ┆ f64        ┆ f64           │
╞══════════╪════════════╪═══════════════╡
│ 6299.0   ┆ 42.0       ┆ 4.0           │
└──────────┴────────────┴───────────────┘


In [58]:
thresholds_f = filtered_company.select([
    pl.col("view_count").quantile(0.95).alias("view_95p"),
    pl.col("follow_count").quantile(0.95).alias("follow_95p"),
    pl.col("reference_count").quantile(0.95).alias("reference_95p")
])
print(thresholds)

shape: (1, 3)
┌──────────┬────────────┬───────────────┐
│ view_95p ┆ follow_95p ┆ reference_95p │
│ ---      ┆ ---        ┆ ---           │
│ f64      ┆ f64        ┆ f64           │
╞══════════╪════════════╪═══════════════╡
│ 6299.0   ┆ 42.0       ┆ 4.0           │
└──────────┴────────────┴───────────────┘


In [54]:
thresholds_z = employee_zero_company.select([
    pl.col("view_count").quantile(0.95).alias("view_95p"),
    pl.col("follow_count").quantile(0.95).alias("follow_95p"),
    pl.col("reference_count").quantile(0.95).alias("reference_95p")
])
print(thresholds)

shape: (1, 3)
┌──────────┬────────────┬───────────────┐
│ view_95p ┆ follow_95p ┆ reference_95p │
│ ---      ┆ ---        ┆ ---           │
│ f64      ┆ f64        ┆ f64           │
╞══════════╪════════════╪═══════════════╡
│ 6299.0   ┆ 42.0       ┆ 4.0           │
└──────────┴────────────┴───────────────┘


In [51]:
# 기준값 추출 (예: 위에서 나온 결과를 직접 변수에 할당)
view_limit = thresholds[0, "view_95p"]
follow_limit = thresholds[0, "follow_95p"]
reference_limit = thresholds[0, "reference_95p"]

company = company.with_columns(
    pl.when((pl.col("view_count") >= view_limit) | (pl.col("follow_count") >= follow_limit) | (pl.col("reference_count") >= reference_limit))
    .then(pl.lit("인기 기업"))
    .otherwise(pl.lit("일반 기업"))
    .alias("company_segment")
)

segment_analysis = (
    company.group_by("company_segment")
    .agg([
        pl.len().alias("count"),
        pl.col("view_count").mean().alias("avg_view"),
        pl.col("follow_count").mean().alias("avg_follow"),
        pl.col("reference_count").mean().alias("avg_reference"),

        # 설립일(found_date)이 문자열이라면 숫자로 바꿔서 평균 설립연도 계산 가능
        # pl.col("found_date").cast(pl.Int64, strict=False).mean().alias("avg_found_year")
    ])
    .sort("avg_view", descending=True)
)

print(segment_analysis)

shape: (2, 5)
┌─────────────────┬───────┬──────────────┬────────────┬───────────────┐
│ company_segment ┆ count ┆ avg_view     ┆ avg_follow ┆ avg_reference │
│ ---             ┆ ---   ┆ ---          ┆ ---        ┆ ---           │
│ str             ┆ u32   ┆ f64          ┆ f64        ┆ f64           │
╞═════════════════╪═══════╪══════════════╪════════════╪═══════════════╡
│ 인기 기업       ┆ 3514  ┆ 10008.666477 ┆ 99.295959  ┆ 6.439385      │
│ 일반 기업       ┆ 38145 ┆ 892.840844   ┆ 1.95871    ┆ 0.216542      │
└─────────────────┴───────┴──────────────┴────────────┴───────────────┘


In [55]:
view_limit_f = thresholds_f[0, "view_95p"]
follow_limit_f = thresholds_f[0, "follow_95p"]
reference_limit_f = thresholds_f[0, "reference_95p"]

filtered_company = filtered_company.with_columns(
    pl.when((pl.col("view_count") >= view_limit_f) | (pl.col("follow_count") >= follow_limit_f) | (pl.col("reference_count") >= reference_limit_f))
    .then(pl.lit("인기 기업"))
    .otherwise(pl.lit("일반 기업"))
    .alias("company_segment")
)

segment_analysis = (
    filtered_company.group_by("company_segment")
    .agg([
        pl.len().alias("count"),
        pl.col("view_count").mean().alias("avg_view"),
        pl.col("follow_count").mean().alias("avg_follow"),
        pl.col("reference_count").mean().alias("avg_reference"),

        # 설립일(found_date)이 문자열이라면 숫자로 바꿔서 평균 설립연도 계산 가능
        # pl.col("found_date").cast(pl.Int64, strict=False).mean().alias("avg_found_year")
    ])
    .sort("avg_view", descending=True)
)

print(segment_analysis)

shape: (2, 5)
┌─────────────────┬───────┬──────────────┬────────────┬───────────────┐
│ company_segment ┆ count ┆ avg_view     ┆ avg_follow ┆ avg_reference │
│ ---             ┆ ---   ┆ ---          ┆ ---        ┆ ---           │
│ str             ┆ u32   ┆ f64          ┆ f64        ┆ f64           │
╞═════════════════╪═══════╪══════════════╪════════════╪═══════════════╡
│ 인기 기업       ┆ 610   ┆ 21937.837705 ┆ 282.834426 ┆ 16.578689     │
│ 일반 기업       ┆ 6071  ┆ 3149.506177  ┆ 20.582276  ┆ 1.795256      │
└─────────────────┴───────┴──────────────┴────────────┴───────────────┘


In [56]:
view_limit_z = thresholds_z[0, "view_95p"]
follow_limit_z = thresholds_z[0, "follow_95p"]
reference_limit_z = thresholds_z[0, "reference_95p"]

employee_zero_company = employee_zero_company.with_columns(
    pl.when((pl.col("view_count") >= view_limit_z) | (pl.col("follow_count") >= follow_limit_z) | (pl.col("reference_count") >= reference_limit_z))
    .then(pl.lit("인기 기업"))
    .otherwise(pl.lit("일반 기업"))
    .alias("company_segment")
)

segment_analysis = (
    employee_zero_company.group_by("company_segment")
    .agg([
        pl.len().alias("count"),
        pl.col("view_count").mean().alias("avg_view"),
        pl.col("follow_count").mean().alias("avg_follow"),
        pl.col("reference_count").mean().alias("avg_reference"),

        # 설립일(found_date)이 문자열이라면 숫자로 바꿔서 평균 설립연도 계산 가능
        # pl.col("found_date").cast(pl.Int64, strict=False).mean().alias("avg_found_year")
    ])
    .sort("avg_view", descending=True)
)

print(segment_analysis)

shape: (2, 5)
┌─────────────────┬───────┬─────────────┬────────────┬───────────────┐
│ company_segment ┆ count ┆ avg_view    ┆ avg_follow ┆ avg_reference │
│ ---             ┆ ---   ┆ ---         ┆ ---        ┆ ---           │
│ str             ┆ u32   ┆ f64         ┆ f64        ┆ f64           │
╞═════════════════╪═══════╪═════════════╪════════════╪═══════════════╡
│ 인기 기업       ┆ 5474  ┆ 3716.621666 ┆ 20.371027  ┆ 1.804165      │
│ 일반 기업       ┆ 29504 ┆ 555.190754  ┆ 0.496407   ┆ 0.0           │
└─────────────────┴───────┴─────────────┴────────────┴───────────────┘


1. 구성원 수와의 상관관계: 인기 기업들은 보통 employee_count가 큰 대기업인가요, 아니면 규모는 작아도 화제성이 높은 스타트업인가?

2. 추천수(reference_count)의 재발견: 평균이 0.7로 매우 낮지만, 인기 기업 그룹 내에서는 이 숫자가 유의미하게 올라가는지 확인해 보세요. 만약 인기 기업에서도 추천수가 낮다면 "유저들은 조용히 지켜만 볼 뿐(조회/팔로우), 직접적인 추천 액션에는 매우 인색하다"는 서비스 이용 패턴 결론을 내릴 수 있습니다.

변수간 선형성을 일부 보임, 낮은 숫자에 데이터가 몰려있어서 적당히 구분지어서 추가적으로 확인해볼 필요있음

In [ ]:
company['cdate'].min(),company['cdate'].max()

기업등록 시점의 최소 최대값
- 등록시점에 따른 기업 평판 (조회,팔로우,추천) 숫자가 다른지 확인해볼 여지있음
- 설립일과 비교를 통해 설립시점으로 부터 해당 플랫폼에 등록한 시점의 차이의 분포를 통해 플랫폼의 평판을 확인해볼 수 있음

# CompanyAddress.csv
- 기업 주소지 정보 테이블	기업id
- 기업 상위 주소지 정보, 주소지명(Ex. 본사)
1. 수도권 (서울 + 경기도): **구**까지 제공  
2. 광역시(수도권 제외): **시**까지 제공  
3. 그 외 지역: **도**까지 제공  
4. 해외 본사: **나라명**까지 제공  

| 컬럼명      | 데이터 타입 | 설명 |
|------------|------------|----------------------------------------------------------------|
| company_id | INTEGER    | 기업 id |
| address1   | STRING     | 주소(상위 정보) |
| name       | STRING     | 주소지명 (Ex: 본사, 연구소 등) |




In [61]:
DATA_DIR = Path(r"C:\Users\user\Desktop\playground\codeit_DA_13th\projects\intermediate1\midproject1\DBsources\topic1")
# 데이터 모아둔 폴더 경로

CompanyAddress = pl.read_parquet(DATA_DIR / 'company.parquet')

CompanyAddress.head()

cdate,mdate,found_date,employee_count,view_count,follow_count,reference_count,company_uuid
datetime[μs],datetime[μs],str,str,i32,i32,i32,str
2022-06-09 04:08:40,2022-06-09 04:08:40,"""""","""0명""",0,0,0,"""c46f2fa5-f940-40fb-bd4a-aa0ff7…"
2017-05-22 10:57:12,2022-07-25 00:12:02,"""""","""0명""",256,0,0,"""725e87bb-de2f-416a-a6b8-1ca8d0…"
2017-11-14 11:42:55,2022-11-04 03:29:16,"""""","""0명""",256,0,0,"""efa3747d-9bbd-4c84-af51-82ccf6…"
2017-11-27 10:59:30,2021-01-17 14:42:06,"""""","""0명""",256,0,0,"""84aa2c20-d0f3-4ec3-ac06-1ef67e…"
2018-01-08 14:41:58,2022-07-25 01:34:16,"""2018-01-08""","""0명""",256,0,0,"""761e76b3-507c-4c8f-b96f-fa47a8…"


In [ ]:
CompanyAddress.select(pl.all().null_count())

cdate,mdate,found_date,employee_count,view_count,follow_count,reference_count,company_uuid
u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0


주소지명 결측치가 일부 존재

In [ ]:
CompanyAddress.company_uuid.value_counts()

1개 기업은 최대 44개의 주소를가짐

In [ ]:
CompanyAddress[CompanyAddress['company_uuid'] =='39db3d5b-4e84-4c52-a68d-c097d859542d']

분점이 많은 경우 동일 uuid에 대해 여러 주소를 가짐

In [ ]:
print(f'주소 존재하는 회사 id 수 : {CompanyAddress.company_uuid.nunique()} , 전체 회사 id 수 { company.company_uuid.nunique()}')

모든 회사id에 대해 주소를 가지는 것은 아님

In [ ]:
CompanyAddress.address.str.split(' ').str[0].value_counts()

주소가 다양하게 들어가 있음 정규표현식등을 활용해서 한글 / 그 외를 구분하고 살펴봐야 할 듯

# CompanyFund.csv
- 기업id, 투자유치일, 투자단계, 투자유치금, 통화(KRW/USD)

| 컬럼명       | 데이터 타입 | 설명       | Comment |
|-------------|------------|------------|----------------------------------------------------------------|
| company_id  | INTEGER    | 기업 id    |  |
| date        | DATE       | 투자일(KST) |  |
| round_type  | STRING     | 투자 단계  | 투자 단계 비공개, Seed, Angel, Series A, Series B, Series C, Series D, Pre-IPO, 해당없음 중 하나 |
| raised      | INTEGER    | 투자 금액   |  |
| currency    | STRING     | 원화/달러   | KRW, USD |


In [ ]:
CompanyFund =pd.read_csv(DATA_DIR +'CompanyFund.csv')
CompanyFund.shape

In [ ]:
CompanyFund.isnull().sum()

In [ ]:
CompanyFund.company_uuid.nunique()

전체회사 41659개 중 투자 관련 정보가 있는 회사는 4785개뿐

In [ ]:
CompanyFund['currency'].unique()

투자 금액은 KRW,USD, nan 타입

In [ ]:
CompanyFund['round_type'].unique()

다양한 투자 단계에 대해 값들이 존재

In [ ]:
CompanyFund[(CompanyFund.raised == 0)].round_type.value_counts()

다양한 round 단계에 대해 투자금액이 0원으로 (미공개?) 있는 경우가 있다. 투자 금액자체를 활용하기에는 누락 데이터가 많아 보임 / round 단계 정도로 파악해야함

# Job.csv
- 채용공고 마스터 테이블
- 용공고id, 기업id, 생성일, 마지막 수정일, 채용분야, 채용시작일, 채용마감일, 경력, 연봉공개여부, 원격근무가능여부

| 컬럼명                | 데이터 타입 | 설명                 | Comment |
|----------------------|------------|----------------------|--------------------------------------------------------------|
| id                  | INTEGER    | 채용공고 id          | PK (job_id) |
| company_id          | INTEGER    | 기업 id              |  |
| cdate              | DATETIME   | 등록 날짜(KST)       |  |
| mdate              | DATETIME   | 마지막 수정일(KST)   | 수정하지 않았을 경우, cdate와 동일 |
| job_field          | STRING     | 업무분야             | SW 개발, 기획/PM, 마케팅, 디자인, 운영, 경영지원, 비즈니스, 투자, HW 개발, 게임 개발 중 하나 |
| career_type_string | STRING     | 경력 형태            | 인턴, 신입, 경력 중 1개 이상 선택 |
| start_date        | DATE       | 채용시작일(KST)      | 등록 날짜와 동일 |
| end_date          | DATE       | 채용마감일(KST)      |  |
| allow_remote      | INTEGER    | 원격근무 가능 여부   | 원격근무 가능 : 1, 원격근무 불가능 : 0 |
| can_show_salary   | INTEGER    | 연봉 공개 여부       | 연봉공개 : 1, 연봉비공개 : 0 |


In [ ]:
job = pd.read_csv(DATA_DIR + 'Job.csv')
display(job.head())
job.shape

In [ ]:
job.mdate.min() , job.mdate.max()

지원 공고 일자 최소 최대, 2013년도는 뭐야..

In [ ]:
job.isnull().sum()

시작,종료일이 없는 경우가 존재, 시작일보다 종료일이 더 없다 -> 시작일은 공고 올린날로 봐도 무방할듯

In [ ]:
pd.to_datetime(job['mdate']).dt.year.value_counts().sort_index().plot(kind = 'bar')
plt.xlabel('년도')
plt.ylabel("공고수")
plt.title('년도별 공고수')
plt.show()

서비스 로그가 22,23년도 뿐인데 ㅂㄷㅂㄷ ;;
19년도 피크 이유와 코로나 시즌에 추세를 따로 구분지어 살펴볼 필요가 있다

In [ ]:
job['job_uuid'].nunique() ==job.shape[0]

job_uuid가 job의 행 숫자와 동일 (모두 유니크하다)

In [ ]:
job['company_uuid'].value_counts()

한 회사에서 많은 공고를 올렸다. 추측해보면
- 특정 정기 채용때 다양한 포지션자리가 오픈
- 여러기간(년도)에 걸쳐 채용 공고를 오픈

확인해볼만한 질문
- 기업들의 채용이 많은 시점은 언제인가?
    - job_filed,  career_type_string 구성에 따라
- 많은 공고가 동시에 열리는 기업의 특성은 무엇인가?
- 경력, 신입, 인턴에 따라 채용공고가 열리는 시점은 차이가 존재하는가?
- 원격 근무와 연봉 근무는 다른 컬럼들과 어떤 관계를 보이는가?
- 직무에 따른 공고의 특성차이가 존재하는지?

In [ ]:
job['career_type_string'].value_counts()

경력공고가 많이 보인다
- 기간별, 코로나시즌, 국제정세(?) 따라 어떤 추세를 보이는지 살펴보면 좋을듯

In [ ]:
job['job_field'].value_counts()

소프트웨어 개발 관련한 공고가 가장 많다

# JobAddress.csv
- 채용공고 주소지 정보 테이블
- 채용공고id, 근무지 상위 주소지 정보, 근무지명

In [ ]:
jobaddress = pd.read_csv(DATA_DIR + 'JobAddress.csv')

In [ ]:
jdu = jobaddress['job_uuid'].nunique()
ju = job['job_uuid'].nunique()
print(f'채용공고 정보의 job uuid 수 : {jdu} ,  채용공고 주소지 정보의 job uuid 수  :  {ju}')

모든 채용공고의 주소지 정보가 나와 있는것은 아니다. 대략 3%만 존재
하나의 job_uuid가 여러개의 jobaddress를 가지는 경우도 존재

In [ ]:
jobaddress[jobaddress.job_uuid =='bf207485-b1cd-4c69-bd91-6e4ac536b197']

- 모든행이 중복인 데이터가 존재, drop_duplicates()로 날리고 봐야함
- 지역,시기별 공고 차이 등에 대해서 분석해볼 수 있음

# JobBookmark.csv
- 채용 북마크 트랜잭션 테이블
- 유저id, 채용공고id, 북마크일시


| 컬럼명      | 데이터 타입 | 설명                | Comment |
|------------|------------|---------------------|---------|
| user_id    | INTEGER    | 유저 id             |  |
| recruit_id | INTEGER    | 북마크한 채용 id    |  |
| cdate      | DATETIME   | 북마크 시점(UTC)    |  |


In [ ]:
jobbookmark  = pd.read_csv(DATA_DIR + 'JobBookmark.csv')
jobbookmark

In [ ]:
jobbookmark['cdate'].min() ,jobbookmark['cdate'].max()

북마크 시점의 최대 최소기간 / 위에서 살펴본 데이터 기간과 정확하게 일치하지 않는다. 특정시점 이후에 북마크 기능이 들어간듯

In [ ]:
jobbookmark.user_uuid.value_counts()

In [ ]:
jobbookmark.user_uuid.value_counts().describe()

user_uuid 관점     
유저가 북마크를 했다는 시점은 취업 의사가 있는 시점으로 추측할 수 있다.    
- 북마크 숫자가 비정상적으로 많은 경우는 크롤러 / 운영진 / 헤비취업자.. 등의 이상치로 보인다.    
- 유저별 북마크 숫자는 평균 7건, 중위수는 3건이다


job_uuid 관점
- 북마크가 많이 된 공고는 많은 유저들이 관심을 가지는 공고로 볼 수 있다.
    - 그러한 공고들이 가지는 특성을 분석해보기
    - 인기 공고들을 선택한 유저의 특성 분석해보기

# Application.csv

- 지원서 마스터 테이블
- 지원서id, 유저id, 채용공고id, 지원일시	현재 하나의 채용공고에 유저가 여러번 지원할 수 있음. 채용담당자가 재지원을 요구하기도 함.

| 컬럼명   | 데이터 타입 | 설명                 | Comment |
|---------|------------|----------------------|---------|
| id      | INTEGER    | 지원 id              |  |
| user_id | INTEGER    | 유저 id              |  |
| job_id  | INTEGER    | 채용 id              |  |
| cdate   | DATETIME   | 지원서 제출시간(UTC) |  |


In [ ]:
ap = pd.read_csv(DATA_DIR + 'Application.csv')
display(ap.head())
ap.shape

- 공고 마감일과 제출일자 사이의 관계 (마감일에 쫓겨 제출하는가?)
- 실제 제출과 북마크는 관계가 있는지
- 채용담당자가 재지원을 요구하는 경우에 대한 케이스 확인
- 같은 회사에 여러번 다른 공고에 지원하는 경우는 얼마나 있는지 / 그러한 회사의 특징은 무엇인지

In [ ]:
ap.cdate.min() , ap.cdate.max()

15년부터 23년까지의 데이터가 존재

In [ ]:
ap.isnull().sum()

결측치는 없다

In [ ]:
ap.application_uuid.value_counts()

application_uuid는 모두 유니크하다 == 지원서는 고유하다.

# log
- log	사용자 로그 테이블
- 유저id, 이벤트 발생 화면경로, 직전 화면경로, 로그 생성 일시, HTTP 요청 메소드, HTTP 응답 코드	"구직자의 로그 데이터만 제공합니다. 로그를 통해 확인 가능한 이벤트는 아래와 같습니다.

```
구직자의 로그 데이터만 제공합니다. 로그를 통해 확인 가능한 이벤트는 아래와 같습니다.

1. 플랫폼 서비스 웹 내에서의 페이지 이동 및 이벤트 전체
*단, 개인정보 이슈상 아래 이벤트를 제외
- 메시지 수신
- 메시지 송신
- 연결신청
- 연결신청 수락 등

2. 채용서비스 관련 각종 이벤트 예시
*모두 1에 포함되는 데이터입니다.
- 채용정보 조회
- 채용 기업 페이지 조회
- 채용 기업의 구성원 프로필 조회
- 지원서 업데이트 (개인정보 이슈상, 어떤 내용이 업데이트되었는지는 미제공)
- 채용공고 북마크
```



| 컬럼명         | 데이터 타입 | 설명               | Comment |
|--------------|------------|------------------|---------|
| user_id      | INTEGER    | 유저 id          |  |
| timestamp    | TIMESTAMP  | 로그 생성 시점   |  |
| date         | DATETIME   | 로그 생성일      |  |
| path         | STRING     | 이벤트 발생 화면 경로 |  |
| response_code | INTEGER    | HTTP 응답 코드   |  |
| method       | STRING     | HTTP 요청 메소드 |  |


In [ ]:
log22 = pd.read_csv(DATA_DIR + 'log_2022.csv').sort_values(['user_uuid','timestamp']).reset_index(drop=True) # 시간순 정렬

In [ ]:
log22.shape

22년 데이터는 천만행

In [ ]:
# 일부 데이터만 필터하는 방법
# chunk = pd.read_csv(directory + 'log_2022.csv',index_col = 0,chunksize=100)
# df= chunk.get_chunk()
# df.head()

In [ ]:
log22.isnull().sum()

3% 결측치 존재

In [ ]:
nulls = log22[log22.URL.isnull()]

In [ ]:
nulls.response_code.value_counts()

결측치 데이터 만의 특성이 따로 있지는 않는것으로 보임

In [ ]:
log22[log22['timestamp'].str.contains('2022-11-13')].sort_values('timestamp') # 특정 timestamp값이 2022-11-13으로 되

In [ ]:
log22['datetime'] = pd.to_datetime(log22['timestamp'].str.replace(' UTC','').map(lambda x: x + ' 00:00:00.000' if len(x) == 10 else (x if '.' in x else x + '.000')))
log22 = log22.drop(columns = ['timestamp'])

In [ ]:
log22.head()

In [ ]:
pd.crosstab(log22['response_code'],log22['method'])

일부 에러가 발생,
- 에러가 유독 많이 발생하는 기간이 있는지? (배포이슈?)
- 특정 api에서 에러가 많이 발생하는지

| 메서드  | 설명                                                                           |
|:------:|:-------------------------------------------------------------------------------|
| **GET**    | 서버에서 **데이터를 조회**(READ)하기 위한 요청. 엔티티 본문을 포함하지 않음.           |
| **HEAD**   | **GET**과 유사하지만, **응답의 헤더 정보만** 받음(본문은 없음). 리소스 존재 여부 확인용. |
| **POST**   | 서버에 **새로운 데이터를 생성**(CREATE)하거나, 데이터를 전송(예: 폼 제출)할 때 사용.    |
| **PUT**    | 서버에 **리소스를 생성/완전히 대체(업데이트)**. 같은 리소스에 여러 번 PUT하면 같은 결과. |
| **DELETE** | 서버의 특정 **리소스를 삭제**.                                                         |

<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>
<br/>


| 코드   | 명칭                        | 설명                                                                       |
|:-----:|:---------------------------|:---------------------------------------------------------------------------|
| **200**  | OK                         | 요청이 성공적으로 처리됨. 일반적으로 **GET**, **POST** 등에 대한 성공 응답.    |
| **301**  | Moved Permanently          | **영구 리다이렉트**. 요청한 리소스가 다른 URL로 영구 이동.                     |
| **302**   | Found (Temporary Redirect) | **임시 리다이렉트**. 요청한 리소스가 임시적으로 다른 URL로 이동.               |
| **400**  | Bad Request                | 클라이언트 측의 잘못된 요청(문법 에러 등). 서버가 요청 해석 불가.               |
| **401**   | Unauthorized               | **인증**(Authentication)이 필요하지만, 유효한 인증 정보가 없음.                |
| **403**   | Forbidden                  | 클라이언트가 **접근 권한**이 없어 요청이 거부됨.                               |
| **404**   | Not Found                  | 요청한 리소스를 찾을 수 없음.                                                 |
| **405**   | Method Not Allowed         | 허용되지 않은 **HTTP 메서드**로 요청.                                         |
| **409**   | Conflict                   | 요청이 서버 상태와 **충돌**이 발생함(중복 데이터, 버전 충돌 등).               |
| **500**   | Internal Server Error      | 서버 내부 에러로 요청 처리 불가. 서버 측에서 문제 발생.                        |
| **503**   | Service Unavailable        | 서버가 현재 **서비스를 처리할 수 없음**(과부하, 점검 등 일시적 장애).         |


<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>
URL기본구조
```<scheme>://<host>/<path1>/<path2>/<path3>/<path-n>?<query>#<fragment>```

In [ ]:
## URL 구조 살펴보기
## 100만개 샘플
## 가이드 코드1
from urllib.parse import urlparse,parse_qs
sample_data = log22['URL'].dropna().sample(1000000,random_state=1).copy().reset_index(drop=True).reset_index(drop=True).to_frame()

parsing_data = pd.DataFrame(list(sample_data['URL'].map(lambda x : urlparse(x)._asdict())))
query_split = sample_data['URL'].map(lambda x : parse_qs(urlparse(x).query))
query_split.name = 'query_dict'

url_total = pd.concat([sample_data,parsing_data,query_split],axis=1)
url_total = url_total.drop(columns =['scheme','netloc','params'])
url_total.head()

```URL기본구조 <scheme>://<host>/<path1>/<path2>/<path3>/<path-n>?<query>#<fragment>```

In [ ]:
pd.set_option('display.max_colwidth',400) # 컬럼폭 늘리는 옵션, 400을 None으로 바꾸면 default 값으로 변경

In [ ]:
## URL 구조 살펴보기
## 전체데이터
from urllib.parse import urlparse,parse_qs
sample_data = log22['URL'].fillna('').reset_index(drop=True).to_frame()

parsing_data = pd.DataFrame(list(sample_data['URL'].map(lambda x : urlparse(x)._asdict())))
query_split = sample_data['URL'].map(lambda x : parse_qs(urlparse(x).query))
query_split.name = 'query_dict'

url_total = pd.concat([sample_data,parsing_data,query_split],axis=1)
url_total = url_total.drop(columns =['scheme','netloc','params'])
url_total.head()

In [ ]:
## url path를 4단계까지 구분하여 누적합 컬럼을 생성
url_total['path_1'] = url_total['path'].map(lambda x : x.split('/')[0] if '/' in x else None)
url_total['path_2'] = url_total['path'].map(lambda x : '/'.join(x.split('/')[:2]) if len(x.split('/')) >=2  else None)
url_total['path_3'] = url_total['path'].map(lambda x : '/'.join(x.split('/')[:3]) if len(x.split('/')) >=3  else None)
url_total['path_4'] = url_total['path'].map(lambda x : '/'.join(x.split('/')[:4]) if len(x.split('/')) >=4  else None)

In [ ]:
log22.head()

In [ ]:
log22_total = pd.concat([log22,url_total.drop(columns =['URL'])],axis=1) # 메모리 엄청 먹음, 일부만 샘플해서 살펴보기를 추천

불필요 변수제거,
메모리 제거

In [ ]:
%xdel url_total

In [ ]:
%xdel log22

In [ ]:
log22_total.shape

In [ ]:
# 30분 기준으로 세션을 생성하여 유저의 행동 분석을 진행하려함
# section_preprocessing_df 데이터 프레임을 생성하여 세션만 부여, 이후 전체데이터에 옮긴 후 해당데이터 삭제
THRESHOLD = pd.Timedelta(minutes=30)
section_preprocessing_df = log22_total[['user_uuid','datetime']].copy()
section_preprocessing_df['time_diff'] = section_preprocessing_df.groupby('user_uuid')['datetime'].diff()
section_preprocessing_df['new_session'] = (section_preprocessing_df['time_diff'] >= THRESHOLD) | (section_preprocessing_df['time_diff'].isna())
section_preprocessing_df['session_id'] = section_preprocessing_df.groupby('user_uuid')['new_session'].cumsum()

In [ ]:
section_preprocessing_df.head(4)

In [ ]:
log22_total['session_id'] = section_preprocessing_df['session_id']

In [ ]:
%xdel section_preprocessing_df

### 세션분석

In [ ]:
sec = log22_total[['user_uuid','datetime','URL','path','session_id']].copy()
sec['session_unique'] = sec['user_uuid'] + '_' + sec['session_id'].astype('str')
sec.head()

In [ ]:
sec['session_unique'].value_counts().describe()

세션의 경우 중위수가 6개의 로그밖에 없다. -> 대체로 무시해도 되는 로그가 많다는 것

In [ ]:
# 세션 샘플 볼 수 있는 코드
# 일부 특징적인 케이스 아래에 별도 표기
smp = sec.sample(1).session_unique.values[0]
print(smp)
session_sample = sec[sec.session_unique == smp]
print(session_sample.shape)
session_sample[['datetime','URL','path','session_id']].reset_index(drop=True)

In [ ]:
def display_session(user_session_unique):
    '''
    세션id 넣으면 세션 로그 출력하는 함수
    '''
    session_sample = sec[sec.session_unique == user_session_unique]
    print(session_sample.shape)
    display(session_sample[['datetime','URL','path','session_id']].reset_index(drop=True))
    return

In [ ]:
# 로그는 검색단어 하나씩 들어갈 때 마다 입력이 됨. suggest가 포함된 URL은 마지막 정보만 있어도 될듯
display_session('83b4586c-1237-4f07-9dff-95353eb1ffe6_4')

In [ ]:
# 실제 지원하는 유저
display_session('4f9fd8a8-1dec-4364-8d83-f780ff5117f4_14')

In [ ]:
# 여러 잡타이틀을 보고 북마크
display_session('c6f0289b-7974-4a08-a912-77dc0636e991_14')

In [ ]:
# 개발자 직무 희망자의 로그
display_session('a3ce9752-16f4-42c6-b797-25770e1aec7b_1')

로그 분석방향 제안
- 로그 자체를 줄일 수 있다
    - 검색의 경우 마지막 입력값에 해당하는 로그만 (suggest, search 등)
    - template 관련한 부분도 불필요해 보임
- 정의된 세션을 통한 유저 행동에 대한 분석 (방문주기, 세션 유지 시간, 행동패턴 등)


api 구조를 보기 위해
- 웹 URL , api 라우터, query string 살펴보기
- 구조를 다양한 방식으로 파싱해서 유저의 행동 추적해보기

In [ ]:
log23 =pd.read_csv(DATA_DIR + 'log_2023.csv',index_col = False)

In [ ]:
log23.shape

23년 데이터는 718만행

In [ ]:
log23.isnull().sum()